# ReAct and Tools

## Setting up DSPy

### Load environment variables

In [1]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

73


### Install Library

In [2]:
# ! uv add dspy
# ! pip install dspy

### Connecting to a language model

DSPy connects to language models with the `dspy.LM` class. To set up a language model, we provide a `"provider/model"` format string and an API key:


In [3]:
import dspy

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-4.1-nano",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

## Give the program tools with `dspy.ReAct`

Giving the program tools allows our program to research and ground its references before writing an answer.

We want to be able to answer a question like:

> "What is the current population of Riyadh, Saudi Arabia in 2026?"

### Introducing: DDGS | Dux Distributed Global Search

[A metasearch library that aggregates results from diverse web search services](https://pypi.org/project/ddgs/).

In [4]:
# ! uv add ddgs
# ! pip install ddgs

In [5]:
from ddgs import DDGS

#### Engines

Here is a list of available search backends one could use:

| DDGS function | Available backends                                                                                           |
| ------------- | :----------------------------------------------------------------------------------------------------------- |
| `text()`        | `bing`, `brave`, `duckduckgo`, `google`, `grokipedia`, `mojeek`, `startpage`, `yandex`, `yahoo`, `wikipedia` |
| `images()`      | `bing`, `duckduckgo`                                                                                         |
| `videos()`      | `duckduckgo`                                                                                                 |
| `news()`        | `bing`, `duckduckgo`, `yahoo`                                                                                |
| `books()`       | `annasarchive`                                                                                               |

Beyond engines, the library has other interesting features. Namely, it includes an optional peer-to-peer distributed cache network. Search results are shared anonymously between users, drastically reducing rate limits and latency for everyone. Checkout the [**DHT Network (BETA)**](https://pypi.org/project/ddgs/).

In [6]:
results = DDGS().text("python programming", max_results=5, backend='wikipedia')

for r in results:
    print("Keys:-")
    print(r.keys())
    break

print()
print("Results:-")
for r in results:
    print(r)

Keys:-
dict_keys(['title', 'href', 'body'])

Results:-
{'title': 'Python (programming language)', 'href': 'https://en.wikipedia.org/wiki/Python_(programming_language)', 'body': 'Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, "plain English" naming, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.

Let's use `ddgs` to search wikipedia

In [7]:
results = DDGS().text(query="Riyadh, Saudi Arabia", backend="bing")

for r in results:
    print("Keys:-")
    print(r.keys())
    break

print()
print("Results:-")
for r in results:
    print(r)

Keys:-
dict_keys(['title', 'href', 'body'])

Results:-
{'title': 'Riyadh, Saudi Arabia', 'href': 'https://en.wikipedia.org/wiki/Riyadh,_Saudi_Arabia', 'body': "Riyadh is the capital and largest city of Saudi Arabia. It is also the capital of the Riyadh Province and the centre of the Riyadh Governorate. Located on the eastern bank of Wadi Hanifa, the current form of the metropolis largely emerged in the 1950s as an expansion of the 18th-century walled town, following the dismantling of its defensive fortifications.It is the largest city on the Arabian Peninsula and is situated in the center of the Nafud desert, on the eastern part of the Najd plateau. The city sits at an average elevation of 600 meters (2,000 ft) above sea level, and receives around 5 million tourists each year, making it the forty-ninth most visited city in the world and the sixth in the Middle East. Riyadh had a population of 7.0 million people in 2022, making it the most populous city in Saudi Arabia, the fifth most 

## Tools are just Python functions

A DSPy tool is a standard Python function, with type-hinted parameters and a docstring. DSPy reads the name, parameters, and docstring of a function to assemble the instructions it sends to an LM.

For example, let’s define a tool that lets an agent search Wikipedia using the `ddgs` library:

In [8]:
def search(query: str) -> list[dict]:
    """Search the web for the given query and return a list of page titles, urls, and bodies."""
    page = DDGS().text(query=query, backend="bing")
    return page

def fetch_page_content(url: str) -> str:
    """Fetch the content of a given URL."""
    result = DDGS().extract(url=url)
    return result['content']

# Test run
# search("Riyadh, Saudi Arabia")

In [9]:
qa_bot = dspy.ReAct("question -> answer", tools=[search, fetch_page_content])

The `dspy.ReAct` module presents it like so:

```text
When selecting the next_tool_name and its next_tool_args, the tool must be one of:
        
(1) search, whose description is <desc>Search the web for the given query and return a list of page titles, urls, and bodies.</desc>. It takes arguments {'query': {'type': 'string'}}.
(2) fetch_page_content, whose description is <desc>Fetch the content of a given URL.</desc>. It takes arguments {'url': {'type': 'string'}}.
(3) finish, whose description is <desc>Marks the task as complete. That is, signals that all information for producing the outputs, i.e. `haiku`, are now available to be extracted.</desc>. It takes arguments {}.
```

Note how DSPy presents the function name, docstring, and parameters to the model. Writing tools, like signatures, requires being mindful about naming. `wikipedia_search` and the parameter `query` are helpful names, that clearly describe their function and role.

Note that there’s a tool in the mix that we didn’t define. `finish` is a special tool used by `dspy.ReAct` that the model calls when it’s done. `dspy.ReAct` populates that one for us.


In [10]:
result = qa_bot(question="What is the current population of Riyadh, Saudi Arabia in 2026?")
print(result.answer)

The current population of Riyadh, Saudi Arabia in 2026 is approximately 7,623,986.


When we call `qa_bot`, we expect it to do the following:

1. Use the `search` function/tool and pass it input rephrased as a search query
2. Decide which of the retunred results is most promising to look into
3. Use the `fetch_page_content` on that
4. Look for the answer to the question within
5. Call `finish`
6. Synthesize the output


## Inspecting the ReAct trajectory

ReAct’s returned `Prediction` instance carries a `trajectory` field: a dictionary that records each thought, tool call, and observation (what the tool returned) in order. When an agent does something surprising, the trajectory is the first thing to read.

We can print it like so:

In [11]:
for step, value in result.trajectory.items():
    print(f"{step}: {value}")

thought_0: Since the population data for Riyadh in 2026 is not readily available and likely involves future projections, I should first search for the latest population figures and any available projections or estimates for 2026 to provide an accurate answer.
tool_name_0: search
tool_args_0: {'query': 'Riyadh population 2026 estimate'}
observation_0: [{'title': 'Saudi Arabia - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Saudi_Arabia', 'body': "Virtually all Saudi inhabitants are Muslim;[418][419][420] as by law Saudi citizens are Muslim. Estimates of the Sunni population range between 85% and 90%, with the remaining 10 to 15% being Shia Muslim,[421][422][423][424] practicing either Twelver Shi'ism or Sulaymani Ismailism."}, {'title': 'Riyadh Population 2026', 'href': 'https://worldpopulationreview.com/cities/saudi-arabia/riyadh', 'body': "Riyadh's 2026 population is now estimated at 7,623,986. In 1992, the population of Riyadh was 2,776,096. Riyadh has grown by 181,214 in the la

Let's make it colorful using the `rich` library (makes terminal output beautiful):

In [ ]:
from utils import print_pretty_react_trajectory

print_pretty_react_trajectory(result)
       

💭 thought_0: Since the population data for Riyadh in 2026 is not readily available and likely involves future 
projections, I should first search for the latest population figures and any available projections or estimates for
2026 to provide an accurate answer.

🛠️ tool_name_0: search

query: Riyadh population 2026 estimate

🔎 observation_0:
[
    {
        'title': 'Saudi Arabia - Wikipedia',
        'href': 'https://en.wikipedia.org/wiki/Saudi_Arabia',
        'body': "Virtually all Saudi inhabitants are Muslim;[418][419][420] as by law Saudi citizens are Muslim. 
Estimates of the Sunni population range between 85% and 90%, with the remaining 10 to 15% being Shia 
Muslim,[421][422][423][424] practicing either Twelver Shi'ism or Sulaymani Ismailism."
    },
    {
        'title': 'Riyadh Population 2026',
        'href': 'https://worldpopulationreview.com/cities/saudi-arabia/riyadh',
        'body': "Riyadh's 2026 population is now estimated at 7,623,986. In 1992, the population of Riyadh was 
2,776,096. Riyadh has grown by 181,214 in the last year, which represents a 2.43% annual change. These population 
values and projections come from Saudi city census series..."
    },
    {
        'title': 'Asia Population (2026) - Worldometer',
        'href': 'https://www.worldometers.info/world-population/asia-population/',
        'body': 'Population: Overall total population (both sexes and all ages) in the region as of July 1 of the 
year indicated, as estimated by the United Nations, Department of Economic and Social Affairs, Population Division.
World Population Prospects: The 2025 Revision.'
    },
    {
        'title': 'World Population by Country in 2026 (World Map) | database.earth',
        'href': 'https://database.earth/population/by-country/2026',
        'body': 'What is the world population projected to be in 2026? By the year 2026, the world population will 
reach a total of 8,300,678,396 people walking the earch.'
    },
    {
        'title': 'Riyadh | Population, Climate, Map, History, & Facts | Britannica',
        'href': 'https://www.britannica.com/place/Riyadh',
        'body': 'Riyadh is Saudi Arabia’s capital and largest city. It became the capital of the Saud dynasty in 
1824 and, except for a brief period in the 19th century and early 20th century, has been the center of Saudi 
government.'
    },
    {
        'title': 'World Population (2026)',
        'href': 'https://populationtoday.com/',
        'body': 'The current state of the world population with real-time data and analysis on population growth, 
demographics, and more.Population today. 8,296,413,515. Thursday, June 11, 2026.'
    },
    {
        'title': 'Top 20 Smallest Countries In The World (Area & Population) 2026',
        'href': 'https://kenyanmagazine.co.ke/smallest-countries-in-the-world-area-population/',
        'body': 'Here are the smallest countries in the world by total area and the smallest countries by 
population, globally. There is a huge difference between a small country by the number...'
    },
    {
        'title': 'Population Pyramid of Iran at 2026 - Population Pyramids',
        'href': 'https://population-pyramid.net/en/pp/iran',
        'body': 'Detailed population distribution of Iran by age and sex of in 2026.'
    },
    {
        'title': 'Malta population (2026) live — Countrymeters',
        'href': 'https://countrymeters.info/en/Malta',
        'body': 'The current population of Malta is 450,097 as of Wednesday, June 10, 2026. Population clock live, 
current, historical and projected population. Births, deaths and migration of population.'
    },
    {
        'title': 'Indiana Population Leaders in 2026 - The State’s 10... - Indiana Hub',
        'href': 'https://indianahub.org/indiana-biggest-cities/',
        'body': 'Indianapolis has no close in-state rival, with a 2026 population estimate of 893,619. Another 
cited count places Indianapolis at 887,642, still far ahead of every other city in Indiana.'
    }
]

💭 thought_1: The search results indicate that there is a specific estimate for the population of Riyadh in 2026, 
which is approximately 7,623,986. Since this information directly answers the question, I should provide this 
estimated population as the answer and conclude the task.

🛠️ tool_name_1: finish

🔎 observation_1: Completed.

The loop finishes by calling the `finish` tool, after which, a **Chain-of-Thought** synthesizes the final answer. We can inspect it with `result.reasoning`:

In [13]:
print(result.reasoning)

Based on the search results, there is a clear estimate provided for the population of Riyadh in 2026, which is approximately 7,623,986. Since the information directly answers the query, I will use this figure as the final answer.


Just to put things back in sequence, here is the final answer again:

In [14]:
print(result.answer)

The current population of Riyadh, Saudi Arabia in 2026 is approximately 7,623,986.


## Control the ReAct agentic loop

ReAct is a test-time (or inference time) loop strategy. We hand the model a set of tools and a task. The `dspy.ReAct` module instructs the model to reason then act using its tools. When the model calls `finish`, DSPy stops the loop and runs one last synthesis pass to produce the declared output fields.

The model decides how many loops to run, but we can cap the number with the `max_iters` parameter, like so:

```py
qa_bot = dspy.ReAct(..., max_iters=4)
```

For tool authoring patterns, MCP integration, and trajectory debugging, see [Tools, ReAct, and MCP](https://dspy.ai/diving-deeper/tools-react-and-mcp/).

## 